# Mini Data Mesh: Dos Dominios con Catálogo Compartido

Objetivo: Construir un mini data mesh de ejemplo con dos dominios (Ventas y Marketing), publicando productos de datos descubiertos a través de un catálogo compartido. Mostramos contratos simples, calidad básica, y una capa semántica ligera para interoperabilidad.

---
## 1. Diseño Lógico
- Dominio Ventas → Producto: orders_daily (dataset de pedidos diarios)
- Dominio Marketing → Producto: campaign_performance (métricas de campañas)
- Catálogo Compartido: índice de productos con metadatos, contratos y etiquetas semánticas
- Capa Semántica: glosario mínimo (Customer, Order, Channel, Campaign)

---
## 2. Contratos de Datos (estructura mínima)
Usaremos un formato Python (dict) para simplificar, con campos, tipos, descripciones, SLAs y reglas de calidad.

In [ ]:
from typing import Dict, Any, List
from dataclasses import dataclass, field
from datetime import datetime
import pandas as pd

@dataclass
class Contract:
    name: str
    schema: List[Dict[str, Any]]
    sla_freshness_minutes: int
    quality_rules: Dict[str, float]
    semantics: Dict[str, str]  # campo -> concepto semántico

@dataclass
class DataProduct:
    domain: str
    name: str
    owner: str
    contract: Contract
    tags: List[str] = field(default_factory=list)
    created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())

class Catalog:
    def __init__(self):
        self.products: Dict[str, DataProduct] = {}
    def register(self, dp: DataProduct):
        key = f

        if key in self.products:
            raise ValueError(
)
        self.products[key] = dp
    def search(self, text: str) -> List[str]:
        text = text.lower()
        return [k for k,dp in self.products.items() if text in k.lower() or any(text in t.lower() for t in dp.tags)]
    def by_concept(self, concept: str) -> List[str]:
        out = []
        for k,dp in self.products.items():
            if concept in dp.contract.semantics.values():
                out.append(k)
        return out
    def show(self):
        return pd.DataFrame([
            {
                'key':k, 'domain':dp.domain, 'name':dp.name, 'owner':dp.owner,
                'freshness_sla_m':dp.contract.sla_freshness_minutes, 'tags':','.join(dp.tags)
            } for k,dp in self.products.items()
        ])

# Definimos dos contratos mínimos
orders_contract = Contract(
    name='orders_daily',
    schema=[
        {'name':'order_id','type':'string','desc':'Identificador de pedido'},
        {'name':'customer_id','type':'string','desc':'Cliente único'},
        {'name':'order_amount','type':'decimal','desc':'Monto del pedido'},
        {'name':'order_date','type':'date','desc':'Fecha'}
    ],
    sla_freshness_minutes=30,
    quality_rules={'completitud':0.99,'duplicados_max':0.001},
    semantics={'customer_id':'Customer','order_amount':'OrderAmount','order_date':'OrderDate'}
)

campaign_contract = Contract(
    name='campaign_performance',
    schema=[
        {'name':'campaign_id','type':'string','desc':'ID campaña'},
        {'name':'channel','type':'string','desc':'Canal'},
        {'name':'impressions','type':'int','desc':'Impresiones'},
        {'name':'clicks','type':'int','desc':'Clicks'},
        {'name':'spend','type':'decimal','desc':'Gasto'}
    ],
    sla_freshness_minutes=45,
    quality_rules={'completitud':0.98,'duplicados_max':0.0005},
    semantics={'channel':'Channel'}
)

ventas_orders = DataProduct(domain='sales', name='orders_daily', owner='owner_sales', contract=orders_contract, tags=['sales','orders','bi','gold'])
marketing_campaign = DataProduct(domain='marketing', name='campaign_performance', owner='owner_marketing', contract=campaign_contract, tags=['marketing','campaign','ads'])

# Registramos en un catálogo compartido
cat = Catalog()
cat.register(ventas_orders)
cat.register(marketing_campaign)
cat.show()

---
## 3. Interoperabilidad vía Capa Semántica Ligera
Consultamos por concepto (e.g., 'Channel') y unimos datasets por un concepto común.

In [ ]:
# Búsqueda por concepto
cat.by_concept('Channel')

In [ ]:
# Simulamos datasets físicos (mini dataframes)
import pandas as pd
orders_df = pd.DataFrame([
    {'order_id':'o1','customer_id':'u1','order_amount':120.0,'order_date':'2025-11-01','channel':'online'},
    {'order_id':'o2','customer_id':'u2','order_amount':75.0,'order_date':'2025-11-01','channel':'store'},
    {'order_id':'o3','customer_id':'u1','order_amount':30.0,'order_date':'2025-11-02','channel':'online'}
])
campaign_df = pd.DataFrame([
    {'campaign_id':'c1','channel':'online','impressions':1000,'clicks':50,'spend':200.0},
    {'campaign_id':'c2','channel':'store','impressions':800,'clicks':32,'spend':120.0}
])
# Unión por concepto 'Channel' (mapeado a campo 'channel')
merged = orders_df.merge(campaign_df, on='channel', how='left')
# KPI ejemplo: ROAS (ingresos / spend) por canal
kpi = merged.groupby('channel').agg(revenue=('order_amount','sum'), spend=('spend','sum'))
kpi['ROAS'] = kpi['revenue']/kpi['spend']
kpi.reset_index()

---
## 4. Checks de Calidad Básicos (como código)
Aplicamos reglas del contrato para validar datasets simulados.

In [ ]:
def check_quality(df: pd.DataFrame, rules: Dict[str, float]):
    results = {}
    if 'completitud' in rules:
        completeness = 1 - (df.isna().any(axis=1).sum()/len(df))
        results['completitud'] = completeness
    if 'duplicados_max' in rules:
        dups = df.duplicated().mean()
        results['duplicados_max'] = dups
    return results

q_orders = check_quality(orders_df, orders_contract.quality_rules)
q_campaign = check_quality(campaign_df, campaign_contract.quality_rules)
q_orders, q_campaign

---
## 5. Publicación y Descubrimiento
- Los dominios publican contratos y metadatos en el catálogo.
- Los consumidores encuentran productos por texto, tags o conceptos semánticos.
- La interoperabilidad se logra al acordar conceptos comunes (glosario) y mapearlos.

### Búsqueda por texto

In [ ]:
cat.search('sales'), cat.search('campaign'), cat.search('orders')

---
## 6. Próximos pasos
- Expandir contratos a formatos serializables (YAML/JSON).
- Integrar validación de esquemas (pydantic/great expectations).
- Hacer *policy-as-code* para acceso y privacidad.
- Usar catálogo real (DataHub/Amundsen) y un *registry* de data products.